In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-24

@author:       Oscar Trevizo
@institution:  Harvard Extension School
@credential:   Graduate Certificate in Data Science (2023)
@context:      Independent project — applying graduate-level concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

data_product_lib — Tutorial Vignette
=====================================

Description:
    A worked tutorial for every class in data_product_lib.py, using a small
    synthetic dataset so the focus stays on the library mechanics rather than
    on any particular domain.

    Classes covered:
        DataProductMetadata  — ownership, provenance, and governance fields
        SemanticLayer        — registry of business-friendly column aliases
        LineageTracker       — append-only log of pipeline transformations
        DataProduct          — wraps a DataFrame with the three above;
                               exposes .schema(), .quality(), .card(), .contract()

    The real-world application of this library is demonstrated in:
        un_wpp_data_product.ipynb  (same folder)

Key Concepts:
    Data Product     — self-describing, governed, discoverable unit of data
    Data Mesh        — organisational pattern; data products are its atomic unit
    Semantic Layer   — maps raw column names to canonical business vocabulary
    Data Contract    — machine-readable agreement on schema, quality, and lineage
    Lineage          — audit trail of every transformation applied to the data

Revision History:
    2026-06-24  Original development
                - Full tutorial: DataProductMetadata, SemanticLayer,
                  LineageTracker, DataProduct
                - Synthetic dataset; no external data files required
                - Quick-reference table and further-reading section
"""

'\nCreated on 2026-06-24\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\ndata_product_lib — Tutorial Vignette\n=====================================\n\nDescription:\n    A worked tutorial for every class in data_product_lib.py, using a small\n    synthetic dataset so the focus stays on the library mechanics rather than\n    on any particular domain.\n\n    Classes covered:\n        DataProductMetadata  — ownership, provenance, and governance fields\n        SemanticLayer        — registry of business-friendly column aliases\n        LineageTracker       — append-only log of pipeline transformations\n        DataProduct          — wraps a DataFrame with the three above;\n                               exposes .schema(), .quality(), .card(), .contract()\n\n    The real-world application 

## Overview

`data_product_lib.py` is a lightweight module that wraps any pandas DataFrame
into a **data product** — a self-describing, governed, and lineage-tracked
artifact that consumers can discover and use without tribal knowledge.

This notebook walks through each class with short, self-contained examples
using a synthetic dataset. No external files are needed.

| Class | Role |
|---|---|
| `DataProductMetadata` | Who owns it, where it came from, what licence covers it |
| `SemanticLayer` | Maps raw column names to business-friendly aliases with units |
| `LineageTracker` | Records every transformation step with input/output shapes |
| `DataProduct` | Assembles the three above around a DataFrame; exposes `.schema()`, `.quality()`, `.card()`, `.contract()` |

> **Real-world example:** `un_wpp_data_product.ipynb` in this folder applies
> all four classes to the UN World Population Prospects 2024 dataset.

In [2]:
import json
import numpy as np
import pandas as pd

from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

---
## Synthetic Dataset

A small fictitious store-sales table — 30 rows, four columns, one deliberate
null — used throughout the tutorial so every output is reproducible.

In [3]:
rng = np.random.default_rng(seed=42)
n   = 30

raw = pd.DataFrame({
    'region'      : rng.choice(['North', 'South', 'East', 'West'], n),
    'product'     : rng.choice(['Widget', 'Gadget', 'Doohickey'], n),
    'year'        : rng.choice([2022, 2023, 2024], n),
    'units_sold'  : rng.integers(10, 500, n),
    'revenue_usd' : rng.uniform(100, 5000, n).round(2),
})

# One deliberate null to make completeness < 100 % for one column
raw.loc[5, 'revenue_usd'] = None

print(f'Shape: {raw.shape}')
raw.head()

Shape: (30, 5)


,region,product,year,units_sold,revenue_usd
0,North,Gadget,2024,223,3375.17
1,West,Widget,2024,404,2408.37
2,East,Widget,2023,422,2869.66
3,South,Gadget,2024,199,3848.49
4,South,Doohickey,2023,450,3210.12


---
## 1. `DataProductMetadata`

A plain dataclass that answers the governance questions every data consumer
needs: *who owns this? where did it come from? is it safe to use?*

| Field | Type | Purpose |
|---|---|---|
| `name` | str | Short identifier for the product |
| `description` | str | One-sentence summary |
| `domain` | str | Business domain (e.g. Sales, Demographics) |
| `owner` | str | Person or team responsible |
| `source` | str | Human-readable source name |
| `source_url` | str | Canonical URL for the source |
| `license` | str | Data licence (e.g. CC BY 4.0, proprietary) |
| `version` | str | Semantic version of this product |
| `refresh_frequency` | str | How often the data is updated |
| `created_at` | str | ISO-8601 timestamp of when this product was built |

In [4]:
from datetime import datetime, timezone

metadata = DataProductMetadata(
    name              = 'Store Sales Demo',
    description       = 'Synthetic store-sales data for data_product_lib tutorial',
    domain            = 'Sales',
    owner             = 'Oscar Trevizo',
    source            = 'Synthetic — generated in-notebook',
    source_url        = '',
    license           = 'N/A',
    version           = '1.0',
    refresh_frequency = 'One-time demo',
    created_at        = datetime.now(timezone.utc).isoformat(),
)

# DataProductMetadata is a plain dataclass — access fields directly
print(metadata.name)
print(metadata.created_at)

Store Sales Demo
2026-06-29T15:20:54.168938+00:00


In [5]:
# Jupyter introspection — use ? or ?? to see the class definition
DataProductMetadata?

Init signature:
DataProductMetadata(
    name: str,
    description: str,
    domain: str,
    owner: str,
    source: str,
    source_url: str,
    license: str,
    version: str,
    refresh_frequency: str,
    created_at: str,
) -> None
Docstring:      DataProductMetadata(name: str, description: str, domain: str, owner: str, source: str, source_url: str, license: str, version: str, refresh_frequency: str, created_at: str)
File:           ~/GitHub/Python/python_vignettes/data_products/data_product_lib.py
Type:           type
Subclasses:     

---
## 2. `SemanticLayer`

A registry that maps a **business name** (stable, human-friendly) to the
underlying **column name** (may change between source versions), its unit,
and a description.

Why this matters: downstream code that references `revenue` instead of
`revenue_usd` keeps working even if the source renames the column.

| Method | Signature | What it does |
|---|---|---|
| `.register()` | `(name, column, unit, description, source_system=None)` | Add one entry |
| `.get()` | `(name) → dict` | Look up one entry by business name |
| `.to_dict()` | `() → dict` | Return the full registry as a plain dict |

In [6]:
semantic = SemanticLayer()

semantic.register('revenue',    'revenue_usd', 'USD',   'Total sales revenue in US dollars')
semantic.register('units',      'units_sold',  'count', 'Number of units sold')
semantic.register('sales_year', 'year',        'year',  'Calendar year of the sale')

# Look up one entry
print(semantic.get('revenue'))

{'column': 'revenue_usd', 'unit': 'USD', 'description': 'Total sales revenue in US dollars', 'source_system': None}


In [7]:
# Full registry as a tidy DataFrame
pd.DataFrame(semantic.to_dict()).T.rename_axis('business_name')

,column,unit,description,source_system
business_name,,,,
revenue,revenue_usd,USD,Total sales revenue in US dollars,None
units,units_sold,count,Number of units sold,None
sales_year,year,year,Calendar year of the sale,None


In [8]:
# Using the semantic layer to slice data without hard-coding column names
entry = semantic.get('revenue')
col   = entry['column']   # resolves to 'revenue_usd'
unit  = entry['unit']

print(f'Business name  : revenue')
print(f'Resolved column: {col}  [{unit}]')
print()
raw[[col]].describe().round(2)

Business name  : revenue
Resolved column: revenue_usd  [USD]



,revenue_usd
count,29.00
mean,2261.53
std,1233.80
min,211.29
25%,1246.30
50%,2363.20
75%,3343.39
max,4281.68


---
## 3. `LineageTracker`

An append-only log. Each call to `.log()` records one pipeline step with its
input shape, output shape, a description of the operation, optional notes,
and a UTC timestamp.

| Method | Signature | What it does |
|---|---|---|
| `.log()` | `(step, operation, input_shape, output_shape, notes=None)` | Append one entry |
| `.to_list()` | `() → list[dict]` | Return all entries as a list of dicts |

In [9]:
lineage = LineageTracker()

# Simulate a two-step pipeline on the synthetic data
lineage.log(
    step         = '1-load',
    operation    = 'generate_synthetic',
    input_shape  = (0, 0),
    output_shape = raw.shape,
    notes        = 'np.random.default_rng(seed=42), n=30',
)

# Simulate a filter step: keep 2023–2024 only
filtered = raw[raw['year'] >= 2023].reset_index(drop=True)
lineage.log(
    step         = '2-filter',
    operation    = 'year_filter',
    input_shape  = raw.shape,
    output_shape = filtered.shape,
    notes        = 'year >= 2023',
)

# Inspect the log
pd.DataFrame(lineage.to_list())

,step,operation,input_shape,output_shape,notes,timestamp
0,1-load,generate_synthetic,"(0, 0)","(30, 5)","np.random.default_rng(seed=42), n=30",2026-06-29T15:21:02.719951
1,2-filter,year_filter,"(30, 5)","(21, 5)",year >= 2023,2026-06-29T15:21:02.721234


---
## 4. `DataProduct`

The top-level class that assembles a DataFrame, `DataProductMetadata`,
`SemanticLayer`, and `LineageTracker` into a single governed artifact.

| Method | What it returns / does |
|---|---|
| `.schema()` | Dict of `{column: dtype}` for every column in the DataFrame |
| `.quality()` | Dict with row count, column count, per-column completeness %, and freshness |
| `.card()` | Prints a formatted human-readable summary (metadata + semantic layer + quality + lineage) |
| `.contract(path)` | Writes all of the above to a JSON sidecar file |

In [10]:
# Assemble — pass the filtered DataFrame and the three supporting objects
dp = DataProduct(filtered, metadata, semantic, lineage)

print(f'Data product : {dp.metadata.name}')
print(f'Rows         : {len(dp.data):,}')
print(f'Semantic     : {len(dp.semantic_layer.to_dict())} entries')
print(f'Lineage      : {len(dp.lineage.to_list())} steps')

Data product : Store Sales Demo
Rows         : 21
Semantic     : 3 entries
Lineage      : 2 steps


In [11]:
# .schema() — column dtypes
pd.DataFrame.from_dict(dp.schema(), orient='index', columns=['dtype']).rename_axis('column')

,dtype
column,
region,object
product,object
year,int64
units_sold,int64
revenue_usd,float64


In [12]:
# .quality() — completeness per column; revenue_usd will be < 100 % if null survived filter
q = dp.quality()
print(f"Row count : {q['row_count']:,}")
print(f"Freshness : {q['freshness']}")
print()
pd.DataFrame.from_dict(
    q['completeness_pct'], orient='index', columns=['completeness_pct']
).rename_axis('column')

Row count : 21
Freshness : 2026-06-29T15:20:54.168938+00:00



,completeness_pct
column,
region,100.0
product,100.0
year,100.0
units_sold,100.0
revenue_usd,100.0


In [13]:
# .card() — the full human-readable summary
dp.card()

  Data Product : Store Sales Demo
  Description  : Synthetic store-sales data for data_product_lib tutorial
  Domain       : Sales
  Owner        : Oscar Trevizo
  Source       : Synthetic — generated in-notebook
  License      : N/A
  Version      : 1.0
  Refresh      : One-time demo
  Created      : 2026-06-29T15:20:54.168938+00:00
  Rows         : 21
  Columns      : 5
  Semantic layer (3 entries):
    revenue                        → revenue_usd  [USD]
    units                          → units_sold  [count]
    sales_year                     → year  [year]
  All columns 100% complete.
  Lineage (2 steps):
    [1-load] generate_synthetic  (0, 0) → (30, 5)  // np.random.default_rng(seed=42), n=30
    [2-filter] year_filter  (30, 5) → (21, 5)  // year >= 2023


In [14]:
import os

os.makedirs('output', exist_ok=True)
contract_path = 'output/store_sales_demo_contract.json'

dp.contract(contract_path)

# Peek at the top-level keys in the JSON
with open(contract_path) as f:
    c = json.load(f)
print('Top-level keys:', list(c.keys()))
print()
print('metadata section:')
print(json.dumps(c['metadata'], indent=2))

Contract written: output/store_sales_demo_contract.json
Top-level keys: ['metadata', 'schema', 'semantic_layer', 'quality', 'lineage']

metadata section:
{
  "name": "Store Sales Demo",
  "description": "Synthetic store-sales data for data_product_lib tutorial",
  "domain": "Sales",
  "owner": "Oscar Trevizo",
  "source": "Synthetic \u2014 generated in-notebook",
  "source_url": "",
  "license": "N/A",
  "version": "1.0",
  "refresh_frequency": "One-time demo",
  "created_at": "2026-06-29T15:20:54.168938+00:00"
}


---
## Quick Reference

Copy-paste starting point for any new data product.

```python
from data_product_lib import DataProductMetadata, SemanticLayer, LineageTracker, DataProduct
from datetime import datetime, timezone

# 1. Metadata
metadata = DataProductMetadata(
    name='My Product', description='...', domain='...', owner='...',
    source='...', source_url='...', license='...', version='1.0',
    refresh_frequency='...', created_at=datetime.now(timezone.utc).isoformat(),
)

# 2. Semantic layer
semantic = SemanticLayer()
semantic.register('business_name', 'raw_column', 'unit', 'description')

# 3. Lineage (log each pipeline step after it runs)
lineage = LineageTracker()
lineage.log('1-load', 'load_source', (0,0), df.shape, notes='optional note')

# 4. Assemble
dp = DataProduct(df, metadata, semantic, lineage)

# 5. Inspect
dp.schema()     # {column: dtype}
dp.quality()    # {row_count, completeness_pct, freshness}
dp.card()       # pretty-print
dp.contract('output/my_product_contract.json')   # write JSON sidecar
```

### Introspection in JupyterLab

```python
DataProduct?       # docstring
DataProduct??      # docstring + source code
DataProduct.card?  # one method's docstring
help(DataProduct)  # plain-Python equivalent, works outside Jupyter
```

---
## Further Reading

`data_product_lib` implements a subset of ideas from the data mesh and data
contract literature. The table below points to the concepts behind each class.

| Concept | Where to read more |
|---|---|
| **Data products & data mesh** | Zhamak Dehghani — *Data Mesh* (O'Reilly, 2022); Martin Fowler's summary at `martinfowler.com/articles/data-mesh-principles.html` |
| **Data contracts** | Chad Sanderson's blog (`dataproducts.substack.com`); the open spec at `datacontract.com` |
| **Semantic / metrics layers** | dbt Semantic Layer docs (`docs.getdbt.com`); also see the Looker / LookML semantic layer pattern |
| **Frictionless Data Table Schema** | A lightweight open standard for column schemas: `specs.frictionlessdata.io/table-schema` |
| **Lineage tools (production-grade)** | OpenLineage (`openlineage.io`); Apache Atlas; DataHub (`datahubproject.io`) |

> **Note:** URLs above are from training knowledge — verify they are current
> before citing them. All are well-established projects as of mid-2026.